# SO-app minimo: Ubuntu pelado + escritorio (una sola celda)

Sistema minimo que arranca siempre. Los programas los instalas tu despues
desde la terminal del escritorio con `apt-get`.
Abre el link `.../vnc.html` en el telefono. Toque = clic.


In [ ]:
import os
import re
import shutil
import socket
import subprocess
import tarfile
import time
import urllib.request

DIR = "/kaggle/working/vmdata/proot"
ROOTFS = DIR + "/rootfs"
VNC_PORT, NOVNC_PORT = 5900, 6080
os.environ["PATH"] = ("/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:"
                      + os.environ.get("PATH", ""))
os.environ["LC_ALL"] = "C"
os.environ["LANG"] = "C"

MARK = os.path.join(ROOTFS, ".soapp-ok")


def log(m=""):
    print(m, flush=True)


def run(cmd, check=False):
    log("$ " + " ".join(cmd))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors="replace")
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError("fallo rc=%d" % p.returncode)
    return p.returncode


def grab(cmd):
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       text=True, errors="replace")
    return r.returncode, r.stdout


def proot(*args):
    return ["/usr/bin/proot", "-R", ROOTFS, "-b", "/dev", "-b", "/proc",
            "-b", "/sys", "-w", "/"] + list(args)


def port_open(port):
    try:
        socket.create_connection(("127.0.0.1", port), timeout=2).close()
        return True
    except OSError:
        return False


APT = ["/usr/bin/apt-get", "-o", "APT::Sandbox::User=root", "-o", "Acquire::ForceIPv4=true"]
TIMEOUT = "/usr/bin/timeout"
os.makedirs(DIR, exist_ok=True)
os.makedirs("/kaggle/working", exist_ok=True)

log("=== [1/4] proot ===")
if not os.path.exists("/usr/bin/proot"):
    log("instalando proot...")
    run(["/usr/bin/apt-get", "update"], check=True)
    run(["/usr/bin/apt-get", "install", "-y", "proot"], check=True)
if not os.path.exists("/usr/bin/proot"):
    raise SystemExit("ERROR: no hay proot ni se pudo instalar")
log("proot OK")

log("=== [2/4] sistema minimo ===")
CLAVES = [os.path.join(ROOTFS, "bin", "sh"),
          os.path.join(ROOTFS, "usr", "bin", "dpkg"),
          os.path.join(ROOTFS, "usr", "bin", "apt-get")]
FALTA = [c for c in CLAVES if not os.path.isfile(c)]
if not FALTA:
    if not os.path.isfile(MARK):
        open(MARK, "w").write("ok\n")
    log("sistema existente, reuso")
else:
    log("sistema incompleto (falta %d archivos clave), reconstruyendo..." % len(FALTA))
    req = urllib.request.Request(
        "https://cdimage.ubuntu.com/ubuntu-base/releases/22.04/release/ubuntu-base-22.04-base-amd64.tar.gz",
        headers={"User-Agent": "soapp/0.1"})
    with urllib.request.urlopen(req, timeout=60) as r, open(os.path.join(DIR, "ubuntu-base.tar.gz"), "wb") as f:
        shutil.copyfileobj(r, f)
    shutil.rmtree(ROOTFS, ignore_errors=True)
    os.makedirs(ROOTFS, exist_ok=True)
    with tarfile.open(os.path.join(DIR, "ubuntu-base.tar.gz")) as t:
        t.extractall(ROOTFS)
    open(MARK, "w").write("ok\n")
    log("sistema minimo OK")
run(proot("/bin/sh", "-c", "dpkg --configure -a 2>&1 | tail -1 || true"))

log("=== [3/4] escritorio (solo 3 paquetes) ===")
run([TIMEOUT, "300"] + proot("/bin/sh", "-c", " ".join(APT + ["update", "-qq"])))
run([TIMEOUT, "600"] + proot("/bin/sh", "-c", " ".join(APT + ["install", "-y", "tigervnc-standalone-server", "openbox", "xterm"]) + " || true"))
for b in ["Xvnc", "openbox-session", "xterm"]:
    rc, _ = grab(proot("/bin/sh", "-c", "command -v " + b))
    if rc != 0:
        raise SystemExit("ERROR: no quedo instalado " + b + " - re-corre la celda")
log("Xvnc + openbox + xterm OK")

NOVNC = os.path.join(DIR, "novnc")
if not os.path.isfile(os.path.join(NOVNC, "vnc.html")):
    log("descargando noVNC...")
    urllib.request.urlretrieve("https://github.com/novnc/noVNC/archive/refs/heads/master.tar.gz", "/tmp/novnc.tgz")
    shutil.rmtree(NOVNC, ignore_errors=True)
    with tarfile.open("/tmp/novnc.tgz") as t:
        t.extractall("/tmp/novnc-x")
    import glob as _g
    shutil.move(_g.glob("/tmp/novnc-x/*/")[0], NOVNC)
else:
    log("noVNC existente, reuso")
if shutil.which("websockify") is None:
    log("instalando websockify...")
    run(["pip", "install", "-q", "websockify"], check=True)

log("=== [4/4] arrancar y tunel ===")
GXE = {"DISPLAY": ":0", "HOME": "/root", "LC_ALL": "C",
       "PATH": "/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"}
run(["/usr/bin/pkill", "-f", "Xvnc :0"])
run(["/usr/bin/pkill", "-f", "websockify.*6080"])
time.sleep(2)
xv = open("/tmp/xvnc.log", "w")
subprocess.Popen(proot("/bin/sh", "-c",
    "rm -f /tmp/.X0-lock; Xvnc :0 -geometry 1280x800 -depth 24 -SecurityTypes None"),
    stdout=xv, stderr=subprocess.STDOUT, start_new_session=True,
    env={**os.environ, **GXE})
ok = False
for _ in range(30):
    if port_open(VNC_PORT):
        ok = True
        break
    time.sleep(2)
if not ok:
    run(["/usr/bin/tail", "-15", "/tmp/xvnc.log"])
    raise SystemExit("ERROR: Xvnc no levanto")
log("Xvnc UP")
subprocess.Popen(proot("/bin/sh", "-c", "DISPLAY=:0 openbox-session"),
                 stdout=open("/tmp/openbox.log", "w"), stderr=subprocess.STDOUT,
                 start_new_session=True, env={**os.environ, **GXE})
time.sleep(2)
subprocess.Popen(proot("/bin/sh", "-c", "DISPLAY=:0 xterm -geometry 100x30"),
                 stdout=open("/tmp/xterm.log", "w"), stderr=subprocess.STDOUT,
                 start_new_session=True, env={**os.environ, **GXE})
log("openbox + terminal lanzados")
ws = open("/tmp/websockify.log", "w")
subprocess.Popen(["websockify", "--web", NOVNC, str(NOVNC_PORT), "127.0.0.1:%d" % VNC_PORT],
                 stdout=ws, stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(3)
if not port_open(NOVNC_PORT):
    run(["/usr/bin/tail", "-15", "/tmp/websockify.log"])
    raise SystemExit("ERROR: websockify no levanto")
log("noVNC UP")
CFD = "/tmp/cloudflared"
if not os.access(CFD, os.X_OK):
    log("descargando cloudflared...")
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CFD)
    os.chmod(CFD, 0o755)
run(["/usr/bin/pkill", "-f", "cloudflared tunnel --url http://127.0.0.1:%d" % NOVNC_PORT])
time.sleep(2)
tlog = open("/tmp/tunnel-vnc.log", "w")
subprocess.Popen([CFD, "tunnel", "--url", "http://127.0.0.1:%d" % NOVNC_PORT],
                 stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True)
URL = ""
for _ in range(45):
    with open("/tmp/tunnel-vnc.log", errors="replace") as f:
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", f.read())
    if m:
        URL = m.group(0)
        break
    time.sleep(2)
if not URL:
    run(["/usr/bin/tail", "-15", "/tmp/tunnel-vnc.log"])
    raise SystemExit("ERROR: sin URL de tunel")
log("ESCRITORIO: " + URL + "/vnc.html")
log("Adentro abre la terminal y instala tus programas con apt-get.")
